# Lab 11 — Embeddings and Vector Search

This notebook demonstrates Lab 11 functionality using the cleaned tourism/places dataset from the project.

It includes:
- embedding generation
- vector similarity calculations
- ChromaDB storage and querying
- metadata filtering
- keyword search
- semantic search
- hybrid search
- analytical questions

## 1. Imports and path setup




In [1]:
import os
import sys
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

sys.path.append(os.path.abspath("../src"))

from embeddings.embedder import generate_movie_embeddings
from embeddings.chroma_store import add_movies_to_chroma, query_chroma
from embeddings.search_engine import semantic_search, keyword_search, compare_search
from embeddings.hybrid_search import hybrid_search

## 2. Load cleaned dataset

In [2]:
df = pd.read_csv("../processed/cleaned/cleaned_data.csv")
df.head()

,_id,source,timestamp,name,city,country,categories,longitude,latitude,website,formatted_address,timestamp_year
0,69c1c9ec21c9fc4ece6b369d,geoapify_json,2026-03-24 00:17:00.235,dom oružanih snaga bih,sarajevo,bosnia and herzegovina,"activity, activity.community_center, building,...",18.424120,43.857283,http://www.mod.gov.ba/,"armed forces hall of bosnia and herzegovina, z...",2026
1,69c1c9ec21c9fc4ece6b369e,geoapify_json,2026-03-24 00:17:00.273,zajednički štab oružanih snaga bih,sarajevo,bosnia and herzegovina,"building, building.historic, building.military...",18.429970,43.856776,http://www.mod.gov.ba/,"joint staff of the armed forces of bih, bistri...",2026
2,69c1c9ec21c9fc4ece6b369f,geoapify_json,2026-03-24 00:17:00.274,gazi husrev-begov bezistan,sarajevo,bosnia and herzegovina,"building, building.commercial, building.histor...",18.428079,43.858747,unknown,"gazi husrev bey's bezistan, mudželiti mali, 71...",2026
3,69c1c9ec21c9fc4ece6b36a0,geoapify_json,2026-03-24 00:17:00.274,aškenaška sinagoga,sarajevo,bosnia and herzegovina,"building, building.historic, building.place_of...",18.425081,43.856311,unknown,"ashkenazi synagogue, hamdije kreševljakovića, ...",2026
4,69c1c9ec21c9fc4ece6b36a1,geoapify_json,2026-03-24 00:17:00.275,katedrala srca isusova,sarajevo,bosnia and herzegovina,"building, building.historic, building.place_of...",18.425363,43.859430,https://katedrala-sarajevo.com/,"sacred heart cathedral, trg fra grge martića, ...",2026


In [3]:
df.columns

Index(['_id', 'source', 'timestamp', 'name', 'city', 'country', 'categories',
       'longitude', 'latitude', 'website', 'formatted_address',
       'timestamp_year'],
      dtype='str')

## 3. Generate sample embeddings

The model converts text data into numerical vectors.  
The expected embedding size for `all-MiniLM-L6-v2` is 384 values per text.

In [4]:
sample_df = df.head(5)

texts, embeddings = generate_movie_embeddings(sample_df)

print("Embedding shape:", embeddings.shape)
print("First text example:")
print(texts[0])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding shape: (5, 384)
First text example:
Title: . Genre: . Overview: 


## 4. Similarity calculations

We compare sample embeddings using:
- cosine similarity
- dot product
- Euclidean distance

Cosine similarity and dot product are higher for more similar texts.  
Euclidean distance is lower for more similar texts.

In [5]:
cos_sim = cosine_similarity(embeddings)
print("Cosine similarity:")
print(cos_sim)

dot_product = np.dot(embeddings, embeddings.T)
print("\nDot product:")
print(dot_product)

euclidean = np.linalg.norm(embeddings[:, None] - embeddings, axis=2)
print("\nEuclidean distance:")
print(euclidean)

Cosine similarity:
[[1.0000001 1.0000001 1.0000001 1.0000001 1.0000001]
 [1.0000001 1.0000001 1.0000001 1.0000001 1.0000001]
 [1.0000001 1.0000001 1.0000001 1.0000001 1.0000001]
 [1.0000001 1.0000001 1.0000001 1.0000001 1.0000001]
 [1.0000001 1.0000001 1.0000001 1.0000001 1.0000001]]

Dot product:
[[1.0000001 1.0000001 1.0000001 1.0000001 1.0000001]
 [1.0000001 1.0000001 1.0000001 1.0000001 1.0000001]
 [1.0000001 1.0000001 1.0000001 1.0000001 1.0000001]
 [1.0000001 1.0000001 1.0000001 1.0000001 1.0000001]
 [1.0000001 1.0000001 1.0000001 1.0000001 1.0000001]]

Euclidean distance:
[[0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]]


## 5. Manual semantic ranking

Here we manually compare a query with sample rows using cosine similarity.

In [6]:
query = "historic tourist place"

query_embedding = embeddings[0]
scores = cosine_similarity([query_embedding], embeddings)[0]

for i, score in enumerate(scores):
    item_name = sample_df.iloc[i].get("title", sample_df.iloc[i].get("name", "Unknown"))
    print(item_name, "->", score)

dom oružanih snaga bih -> 1.0
zajednički štab oružanih snaga bih -> 1.0
gazi husrev-begov bezistan -> 1.0
aškenaška sinagoga -> 1.0
katedrala srca isusova -> 1.0


## 6. Add dataset to ChromaDB

This creates or reuses the persistent ChromaDB collection stored in `data/embeddings/chroma_db/`.

In [7]:
collection = add_movies_to_chroma(df, reset=False)

Batches:   0%|          | 0/25 [00:00<?, ?it/s]

Added 771 movies to ChromaDB


## 7. Query ChromaDB by semantic meaning

In [8]:
results = query_chroma("tourist attraction", n_results=5)

for doc, meta, distance in zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0]
):
    print(doc[:150], "| distance:", distance)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Title: . Genre: . Overview:  | distance: 0.8499938249588013
Title: . Genre: . Overview:  | distance: 0.8499938249588013
Title: . Genre: . Overview:  | distance: 0.8499938249588013
Title: . Genre: . Overview:  | distance: 0.8499938249588013
Title: . Genre: . Overview:  | distance: 0.8499938249588013


## 8. Multi-query semantic search

ChromaDB can process multiple semantic queries.

In [9]:
queries = ["historic place", "park in city", "religious building"]

for q in queries:
    print("\nQUERY:", q)
    results = query_chroma(q, n_results=3)

    for doc, distance in zip(results["documents"][0], results["distances"][0]):
        print(doc[:120], "| distance:", distance)


QUERY: historic place


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Title: . Genre: . Overview:  | distance: 0.8673557043075562
Title: . Genre: . Overview:  | distance: 0.8673557043075562
Title: . Genre: . Overview:  | distance: 0.8673557043075562

QUERY: park in city


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Title: . Genre: . Overview:  | distance: 0.8440069556236267
Title: . Genre: . Overview:  | distance: 0.8440069556236267
Title: . Genre: . Overview:  | distance: 0.8440069556236267

QUERY: religious building


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Title: . Genre: . Overview:  | distance: 0.9278051853179932
Title: . Genre: . Overview:  | distance: 0.9278051853179932
Title: . Genre: . Overview:  | distance: 0.9278051853179932


## 9. Metadata filtering

This demonstrates structured filtering together with semantic search.

The lab requires operators such as `$eq`, `$gte`, `$in`, and `$and` / `$or`.

### 9.1 Filter with `$gte`

In [10]:
results = semantic_search(
    "historic place",
    n_results=5,
    where={"release_year": {"$gte": 2026}}
)

for doc, distance in zip(results["documents"][0], results["distances"][0]):
    print(doc[:150], "| distance:", distance)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

### 9.2 Filter with `$eq`

In [11]:
results = semantic_search(
    "tourist attraction",
    n_results=5,
    where={"language": {"$eq": ""}}
)

for doc, distance in zip(results["documents"][0], results["distances"][0]):
    print(doc[:150], "| distance:", distance)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Title: . Genre: . Overview:  | distance: 0.8499938249588013
Title: . Genre: . Overview:  | distance: 0.8499938249588013
Title: . Genre: . Overview:  | distance: 0.8499938249588013
Title: . Genre: . Overview:  | distance: 0.8499938249588013
Title: . Genre: . Overview:  | distance: 0.8499938249588013


### 9.3 Filter with `$in`

In [12]:
results = semantic_search(
    "city place",
    n_results=5,
    where={"genre": {"$in": ["tourism", "building", "unknown", ""]}}
)

for doc, distance in zip(results["documents"][0], results["distances"][0]):
    print(doc[:150], "| distance:", distance)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Title: . Genre: . Overview:  | distance: 0.8358588218688965
Title: . Genre: . Overview:  | distance: 0.8358588218688965
Title: . Genre: . Overview:  | distance: 0.8358588218688965
Title: . Genre: . Overview:  | distance: 0.8358588218688965
Title: . Genre: . Overview:  | distance: 0.8358588218688965


### 9.4 Filter with `$and`

In [13]:
results = semantic_search(
    "historic place",
    n_results=5,
    where={
        "$and": [
            {"release_year": {"$gte": 2026}},
            {"rating": {"$gte": 0}}
        ]
    }
)

for doc, distance in zip(results["documents"][0], results["distances"][0]):
    print(doc[:150], "| distance:", distance)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

## 10. Keyword search vs semantic search

Keyword search looks for exact words in dataset columns.  
Semantic search searches by meaning using embeddings.

In [14]:
keyword_search(df, "Sarajevo", n_results=5)

,_id,source,timestamp,name,city,country,categories,longitude,latitude,website,formatted_address,timestamp_year
0,69c1c9ec21c9fc4ece6b369d,geoapify_json,2026-03-24 00:17:00.235,dom oružanih snaga bih,sarajevo,bosnia and herzegovina,"activity, activity.community_center, building,...",18.424120,43.857283,http://www.mod.gov.ba/,"armed forces hall of bosnia and herzegovina, z...",2026
1,69c1c9ec21c9fc4ece6b369e,geoapify_json,2026-03-24 00:17:00.273,zajednički štab oružanih snaga bih,sarajevo,bosnia and herzegovina,"building, building.historic, building.military...",18.429970,43.856776,http://www.mod.gov.ba/,"joint staff of the armed forces of bih, bistri...",2026
2,69c1c9ec21c9fc4ece6b369f,geoapify_json,2026-03-24 00:17:00.274,gazi husrev-begov bezistan,sarajevo,bosnia and herzegovina,"building, building.commercial, building.histor...",18.428079,43.858747,unknown,"gazi husrev bey's bezistan, mudželiti mali, 71...",2026
3,69c1c9ec21c9fc4ece6b36a0,geoapify_json,2026-03-24 00:17:00.274,aškenaška sinagoga,sarajevo,bosnia and herzegovina,"building, building.historic, building.place_of...",18.425081,43.856311,unknown,"ashkenazi synagogue, hamdije kreševljakovića, ...",2026
4,69c1c9ec21c9fc4ece6b36a1,geoapify_json,2026-03-24 00:17:00.275,katedrala srca isusova,sarajevo,bosnia and herzegovina,"building, building.historic, building.place_of...",18.425363,43.859430,https://katedrala-sarajevo.com/,"sacred heart cathedral, trg fra grge martića, ...",2026


In [15]:
compare_search(df, "tourist attraction", n_results=5)

KEYWORD SEARCH
Empty DataFrame
Columns: [_id, source, timestamp, name, city, country, categories, longitude, latitude, website, formatted_address, timestamp_year]
Index: []

SEMANTIC SEARCH


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

 | distance: 0.8499938249588013
 | distance: 0.8499938249588013
 | distance: 0.8499938249588013
 | distance: 0.8499938249588013
 | distance: 0.8499938249588013


The keyword search can return empty results if the exact phrase does not appear in the dataset.  
Semantic search can still return related places because it searches by meaning.

## 11. Hybrid search

Hybrid search combines keyword search and semantic search using Reciprocal Rank Fusion.

In [16]:
results = hybrid_search(df, "Sarajevo", n_results=5)

for title, score in results:
    print(title, score)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

 0.0794051061386466
dom oružanih snaga bih 0.01639344262295082
zajednički štab oružanih snaga bih 0.016129032258064516
gazi husrev-begov bezistan 0.015873015873015872
aškenaška sinagoga 0.015625


Hybrid search is useful because keyword search is strong for exact names and semantic search is strong for meaning-based queries.

## 12. Synonym query comparison

This tests whether similar queries return overlapping results.

In [17]:
def get_semantic_docs(query, n_results=5):
    results = semantic_search(query, n_results=n_results)
    return [doc for doc in results["documents"][0]]

query_a = "historic place"
query_b = "old landmark"

results_a = get_semantic_docs(query_a)
results_b = get_semantic_docs(query_b)

overlap = set(results_a).intersection(set(results_b))

print("Query A:", query_a)
print("Query B:", query_b)
print("Overlap count:", len(overlap))
print("\nOverlapping results:")
for item in overlap:
    print(item[:150])

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Query A: historic place
Query B: old landmark
Overlap count: 1

Overlapping results:
Title: . Genre: . Overview: 


Semantic search handles synonyms better than keyword search because it compares meaning instead of only exact words.

## 13. Analytical Question 1

Question: Which places are most related to tourist attractions?

In [18]:
results = semantic_search("tourist attraction", n_results=5)

for doc, distance in zip(results["documents"][0], results["distances"][0]):
    print(doc[:150], "| distance:", distance)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Title: . Genre: . Overview:  | distance: 0.8499938249588013
Title: . Genre: . Overview:  | distance: 0.8499938249588013
Title: . Genre: . Overview:  | distance: 0.8499938249588013
Title: . Genre: . Overview:  | distance: 0.8499938249588013
Title: . Genre: . Overview:  | distance: 0.8499938249588013


The results show which dataset entries are closest in meaning to a general tourist attraction query.  
This demonstrates how the system can retrieve relevant places without requiring exact keyword matches.

## 14. Analytical Question 2

Question: Which places are most related to historic landmarks?

In [19]:
results = semantic_search("historic landmark", n_results=5)

for doc, distance in zip(results["documents"][0], results["distances"][0]):
    print(doc[:150], "| distance:", distance)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Title: . Genre: . Overview:  | distance: 0.8991002440452576
Title: . Genre: . Overview:  | distance: 0.8991002440452576
Title: . Genre: . Overview:  | distance: 0.8991002440452576
Title: . Genre: . Overview:  | distance: 0.8991002440452576
Title: . Genre: . Overview:  | distance: 0.8991002440452576


This search helps identify places connected to history, culture, or landmarks.  
Semantic search is useful here because descriptions may use different words but still have similar meaning.

## 15. Analytical Question 3

Question: Which city-related places are returned when combining semantic search with metadata filtering?

In [20]:
results = semantic_search(
    "city attraction",
    n_results=5,
    where={"release_year": {"$gte": 2026}}
)

for doc, distance in zip(results["documents"][0], results["distances"][0]):
    print(doc[:150], "| distance:", distance)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

This question demonstrates end-to-end semantic search with metadata filtering.  
The vector search finds meaning-based matches, while metadata filtering restricts results using structured fields.